# Butterfly Diffusion - Kaggle GPU Notebook

## Install

In [ ]:
# Kaggle pre-installs PyTorch + CUDA. Detect the assigned GPU first so that
# if a pre-Volta card (P100, sm_60) is allocated we can swap to a torch build
# that still supports it. T4 and newer use the default torch unchanged.
import subprocess, sys

def _gpu_cap():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader,nounits"],
            text=True, timeout=10).strip().splitlines()[0]
        return float(out)
    except Exception as e:
        print(f"nvidia-smi probe failed: {e}"); return None

_cap = _gpu_cap()
print(f"Detected GPU compute capability: {_cap}")

if _cap is not None and _cap < 7.0:
    print(f"Pre-Volta GPU (sm_{int(_cap*10):02d}); installing torch 2.4.1 with sm_60 support...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu121"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "datasets", "diffusers", "einops", "imageio[ffmpeg]"], check=False)
print("Dependencies ready.")


## GPU Diagnostics

In [ ]:
import subprocess, torch, sys, platform
print("=" * 60)
print(" GPU DIAGNOSTICS")
print("=" * 60)
print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.platform()}")
print(f"Torch       : {torch.__version__}")
print(f"CUDA build  : {torch.version.cuda}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "GPU runtime is required. Enable Accelerator -> GPU T4 in Kaggle."
props = torch.cuda.get_device_properties(0)
print(f"Device      : {props.name}")
print(f"Capability  : {props.major}.{props.minor}")
print(f"VRAM total  : {props.total_memory/1e9:.2f} GB")
print(f"SM count    : {props.multi_processor_count}")
print("-" * 60)
try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
                                   "--format=csv,noheader"], text=True).strip()
    print("nvidia-smi  :", out)
except Exception as e:
    print(f"nvidia-smi unavailable: {e}")
print("=" * 60)


## Imports

In [ ]:
import os, math, json, time, random, copy, warnings, shutil, zipfile
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional, Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler

import matplotlib.pyplot as plt
import seaborn as sns
import imageio
from einops import rearrange
from tqdm.auto import tqdm
from datasets import load_dataset
from torchvision import transforms
from torchvision.utils import make_grid

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "#0e1117", "axes.facecolor": "#0e1117",
    "savefig.facecolor": "#0e1117", "axes.labelcolor": "#e6e6e6",
    "xtick.color": "#cfcfcf", "ytick.color": "#cfcfcf",
    "axes.titlecolor": "#ffffff", "text.color": "#e6e6e6",
    "axes.edgecolor": "#3a3a3a", "grid.color": "#2a2a2a",
})

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

# Auto-select AMP dtype based on GPU capability (bf16 on Ampere+, fp16 elsewhere).
_cap = torch.cuda.get_device_capability(0)
AMP_DTYPE = torch.bfloat16 if _cap[0] >= 8 else torch.float16
print(f"AMP dtype   : {AMP_DTYPE}")


## Working Directories

In [ ]:
# Kaggle working directory is the only writable + downloadable location.
KAGGLE_ROOT  = Path("/kaggle/working")
CKPT_DIR     = KAGGLE_ROOT / "checkpoints"
OUT_DIR      = KAGGLE_ROOT / "outputs"
ASSETS_DIR   = KAGGLE_ROOT / "assets"
SAMPLES_DIR  = KAGGLE_ROOT / "generated_samples"
GIFS_DIR     = OUT_DIR / "gifs"
TRAJ_DIR     = OUT_DIR / "trajectories"
EPOCH_DIR    = OUT_DIR / "samples"

for p in [CKPT_DIR, OUT_DIR, ASSETS_DIR, SAMPLES_DIR, GIFS_DIR, TRAJ_DIR, EPOCH_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Working directories:")
for p in [CKPT_DIR, OUT_DIR, ASSETS_DIR, SAMPLES_DIR, GIFS_DIR, TRAJ_DIR, EPOCH_DIR]:
    print(f"  {p}")


## Config

In [ ]:
# ----------------------------------------------------------------------
# MODE switch.  Set MODE = "smoke" for a 2-epoch sanity run (~2-3 min).
#              Set MODE = "full"   for the production-quality 80-epoch run.
# ----------------------------------------------------------------------
MODE = "full"     # "smoke" | "full"

@dataclass
class Config:
    seed: int = 42

    image_size: int = 64
    in_channels: int = 3
    base_channels: int = 64
    channel_mult: Tuple[int, ...] = (1, 2, 4, 8)
    num_res_blocks: int = 2
    attn_resolutions: Tuple[int, ...] = (16, 8)
    time_embed_dim: int = 256
    dropout: float = 0.1

    timesteps: int = 1000
    beta_schedule: str = "cosine"
    beta_start: float = 1e-4
    beta_end: float = 0.02

    batch_size: int = 64
    num_workers: int = 2
    epochs: int = 80
    lr: float = 2e-4
    weight_decay: float = 1e-6
    grad_clip: float = 1.0
    warmup_steps: int = 500
    ema_decay: float = 0.9999
    use_amp: bool = True

    sample_every: int = 5
    n_samples: int = 16
    ddim_steps: int = 50

    resume: bool = True       # auto-resume from CKPT_DIR/last.pt if present


cfg = Config()

if MODE == "smoke":
    cfg.epochs = 2
    cfg.sample_every = 1
    cfg.ddim_steps = 20
    cfg.warmup_steps = 50
    cfg.n_samples = 8
    print(">>> SMOKE TEST MODE <<<  (epochs=2, ddim_steps=20)")
elif MODE == "full":
    print(">>> FULL TRAINING MODE <<<")
else:
    raise ValueError(f"Unknown MODE: {MODE}")


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)
print(json.dumps(asdict(cfg), indent=2))


## Data

In [ ]:
ds = load_dataset("huggan/smithsonian_butterflies_subset", split="train")
print(f"Dataset images: {len(ds)}")

tfm = transforms.Compose([
    transforms.Resize(cfg.image_size, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(cfg.image_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])


def collate(batch):
    imgs = torch.stack([tfm(b["image"].convert("RGB")) for b in batch], dim=0)
    return imgs.contiguous(memory_format=torch.channels_last)


# Kaggle: num_workers > 2 often causes /dev/shm issues; keep modest.
loader = DataLoader(
    ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=cfg.num_workers, pin_memory=True,
    persistent_workers=cfg.num_workers > 0,
    prefetch_factor=4 if cfg.num_workers > 0 else None,
    drop_last=True, collate_fn=collate,
)
print(f"Batches/epoch: {len(loader)}")


## Dataset Preview

In [ ]:
def denorm(x): return (x.clamp(-1, 1) + 1) / 2


@torch.no_grad()
def show_grid(imgs, nrow=8, title=None, save=None, figsize=(10, 10)):
    grid = make_grid(denorm(imgs.float().cpu()), nrow=nrow, padding=2)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(grid.permute(1, 2, 0).numpy()); ax.axis("off")
    if title: ax.set_title(title, fontsize=14, pad=10)
    plt.tight_layout()
    if save:
        Path(save).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()


batch = next(iter(loader))
show_grid(batch[:64], nrow=8, title="Smithsonian Butterflies - Real Samples",
          save=str(ASSETS_DIR / "dataset_grid.png"))


## Noise Scheduler

In [ ]:
def linear_beta_schedule(T, b0, b1):
    return torch.linspace(b0, b1, T, dtype=torch.float32)


def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    x = torch.linspace(0, T, steps, dtype=torch.float64)
    a = torch.cos(((x / T) + s) / (1 + s) * math.pi / 2) ** 2
    a = a / a[0]
    betas = 1 - (a[1:] / a[:-1])
    return betas.clamp(1e-5, 0.999).float()


class Diffusion:
    def __init__(self, cfg: Config, device):
        self.T = cfg.timesteps
        betas = (cosine_beta_schedule(self.T) if cfg.beta_schedule == "cosine"
                 else linear_beta_schedule(self.T, cfg.beta_start, cfg.beta_end))
        alphas = 1.0 - betas
        ac = torch.cumprod(alphas, dim=0)
        ac_prev = F.pad(ac[:-1], (1, 0), value=1.0)
        self.betas = betas.to(device)
        self.alphas = alphas.to(device)
        self.alphas_cum = ac.to(device)
        self.alphas_cum_prev = ac_prev.to(device)
        self.sqrt_alphas_cum = torch.sqrt(ac).to(device)
        self.sqrt_one_minus_alphas_cum = torch.sqrt(1 - ac).to(device)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas).to(device)
        self.posterior_var = (betas * (1 - ac_prev) / (1 - ac)).to(device)

    def q_sample(self, x0, t, noise=None):
        if noise is None: noise = torch.randn_like(x0)
        sa = self.sqrt_alphas_cum.gather(0, t).view(-1, 1, 1, 1)
        sm = self.sqrt_one_minus_alphas_cum.gather(0, t).view(-1, 1, 1, 1)
        return sa * x0 + sm * noise, noise


diff = Diffusion(cfg, DEVICE)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(diff.betas.cpu(), color="#ff6ec7", linewidth=2); ax[0].set_title(r"$\beta_t$")
ax[1].plot(diff.alphas_cum.cpu(), color="#7df9ff", linewidth=2); ax[1].set_title(r"$\bar{\alpha}_t$")
for a in ax: a.set_xlabel("t")
plt.tight_layout(); plt.savefig(ASSETS_DIR / "schedule.png", dpi=150, bbox_inches="tight"); plt.show()


## Forward Diffusion

In [ ]:
@torch.no_grad()
def visualize_forward(x0, steps=(0, 50, 100, 250, 500, 750, 999)):
    x0 = x0.to(DEVICE); rows = []
    for tv in steps:
        t = torch.full((x0.size(0),), tv, device=DEVICE, dtype=torch.long)
        xt, _ = diff.q_sample(x0, t); rows.append(xt.cpu())
    cat = torch.cat([r[:8] for r in rows], dim=0)
    show_grid(cat, nrow=8, figsize=(14, len(steps) * 1.8),
              title=f"Forward Diffusion  t = {list(steps)}",
              save=str(ASSETS_DIR / "forward_process.png"))


visualize_forward(batch[:8])


## Time Embeddings

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim): super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


## U-Net

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim, dropout=0.1, groups=32):
        super().__init__()
        gi, go = min(groups, in_ch), min(groups, out_ch)
        self.norm1 = nn.GroupNorm(gi, in_ch); self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(t_dim, out_ch)
        self.norm2 = nn.GroupNorm(go, out_ch); self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)


class AttnBlock(nn.Module):
    def __init__(self, ch, heads=4, groups=32):
        super().__init__()
        self.norm = nn.GroupNorm(min(groups, ch), ch); self.heads = heads
        self.qkv = nn.Conv2d(ch, ch * 3, 1); self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv(self.norm(x))
        q, k, v = rearrange(qkv, "b (three heads c) h w -> three b heads (h w) c",
                            three=3, heads=self.heads)
        attn = ((q @ k.transpose(-2, -1)) * (q.size(-1) ** -0.5)).softmax(dim=-1)
        out = rearrange(attn @ v, "b heads (h w) c -> b (heads c) h w", h=h, w=w)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.op = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.op(x)


class Upsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.op = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x): return self.op(F.interpolate(x, scale_factor=2, mode="nearest"))


class UNet(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        C, mults, nb = cfg.base_channels, cfg.channel_mult, cfg.num_res_blocks
        t_dim = cfg.time_embed_dim
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(C), nn.Linear(C, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim),
        )
        self.in_conv = nn.Conv2d(cfg.in_channels, C, 3, padding=1)

        chs = [C]; cur = C; res = cfg.image_size
        self.down_blocks = nn.ModuleList()
        for i, m in enumerate(mults):
            oc = C * m
            for _ in range(nb):
                blk = nn.ModuleList([ResBlock(cur, oc, t_dim, cfg.dropout)])
                if res in cfg.attn_resolutions: blk.append(AttnBlock(oc))
                self.down_blocks.append(blk); cur = oc; chs.append(cur)
            if i != len(mults) - 1:
                self.down_blocks.append(nn.ModuleList([Downsample(cur)]))
                chs.append(cur); res //= 2

        self.mid = nn.ModuleList([
            ResBlock(cur, cur, t_dim, cfg.dropout),
            AttnBlock(cur),
            ResBlock(cur, cur, t_dim, cfg.dropout),
        ])

        self.up_blocks = nn.ModuleList()
        for i, m in enumerate(reversed(mults)):
            oc = C * m
            for _ in range(nb + 1):
                sk = chs.pop()
                blk = nn.ModuleList([ResBlock(cur + sk, oc, t_dim, cfg.dropout)])
                if res in cfg.attn_resolutions: blk.append(AttnBlock(oc))
                self.up_blocks.append(blk); cur = oc
            if i != len(mults) - 1:
                self.up_blocks.append(nn.ModuleList([Upsample(cur)])); res *= 2

        self.out_norm = nn.GroupNorm(min(32, cur), cur)
        self.out_conv = nn.Conv2d(cur, cfg.in_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = self.in_conv(x); skips = [h]
        for blk in self.down_blocks:
            first = blk[0]
            if isinstance(first, Downsample):
                h = first(h); skips.append(h)
            else:
                h = first(h, t_emb)
                for L in list(blk)[1:]: h = L(h)
                skips.append(h)
        for L in self.mid:
            h = L(h, t_emb) if isinstance(L, ResBlock) else L(h)
        for blk in self.up_blocks:
            first = blk[0]
            if isinstance(first, Upsample):
                h = first(h)
            else:
                h = torch.cat([h, skips.pop()], dim=1)
                h = first(h, t_emb)
                for L in list(blk)[1:]: h = L(h)
        return self.out_conv(F.silu(self.out_norm(h)))


model = UNet(cfg).to(DEVICE, memory_format=torch.channels_last)
n_params = sum(p.numel() for p in model.parameters())
print(f"U-Net parameters: {n_params/1e6:.2f}M")


## EMA

In [ ]:
class EMA:
    """Karras-style EMA with warmup.
    Effective decay at step t is min(target_decay, (1+t)/(10+t)) so the
    shadow tracks the live model fast at the start and asymptotes to the
    high target decay once enough steps have accumulated.
    """
    def __init__(self, model, target_decay=0.9999):
        self.target = target_decay
        self.step = 0
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.target, (1 + self.step) / (10 + self.step))
        for ep, p in zip(self.shadow.parameters(), model.parameters()):
            ep.data.mul_(d).add_(p.data, alpha=1 - d)

    def state_dict(self): return self.shadow.state_dict()
    def load_state_dict(self, sd): self.shadow.load_state_dict(sd)


ema = EMA(model, cfg.ema_decay)


## Training Utilities

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                              betas=(0.9, 0.999), weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(loader)


def lr_lambda(step):
    if step < cfg.warmup_steps: return step / max(1, cfg.warmup_steps)
    progress = (step - cfg.warmup_steps) / max(1, total_steps - cfg.warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
# GradScaler only needed for fp16; with bf16 we skip scaling.
scaler = GradScaler(enabled=(cfg.use_amp and AMP_DTYPE == torch.float16))


def gpu_mem_mb(): return torch.cuda.memory_allocated() / 1e6


def save_checkpoint(path, epoch, loss):
    torch.save({
        "model": model.state_dict(),
        "ema": ema.state_dict(),
        "optim": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch, "loss": loss,
        "config": asdict(cfg),
    }, path)


def try_resume():
    ckpt_path = CKPT_DIR / "last.pt"
    if not cfg.resume or not ckpt_path.exists():
        return 0
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"]); ema.load_state_dict(ckpt["ema"])
    optimizer.load_state_dict(ckpt["optim"]); scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    start = ckpt["epoch"]
    print(f"Resumed from epoch {start} (loss={ckpt['loss']:.4f})")
    return start


start_epoch = try_resume()


## Samplers

In [ ]:
# Sampling intentionally runs in fp32 (no autocast). High-t reverse steps
# produce intermediate activations that exceed fp16 dynamic range (>65504),
# which would NaN the U-Net forward. Training still uses fp16 autocast.
# Both samplers use the numerically-stable x0-clamped posterior formula
# rather than the algebraically-equivalent eps formula, which amplifies
# prediction errors by 1/sqrt(alpha_t) (~31x at the cosine-schedule tail).

@torch.no_grad()
def ddpm_sample(net, n, return_trajectory=False):
    net.eval()
    x = torch.randn(n, cfg.in_channels, cfg.image_size, cfg.image_size,
                    device=DEVICE)
    traj = []
    snap_every = max(1, cfg.timesteps // 20)
    for i in range(cfg.timesteps - 1, -1, -1):
        t = torch.full((n,), i, device=DEVICE, dtype=torch.long)
        eps = net(x, t).float()
        sac  = diff.sqrt_alphas_cum[i]
        smac = diff.sqrt_one_minus_alphas_cum[i]
        x0 = ((x - smac * eps) / sac).clamp(-1, 1)
        coef_x0 = (torch.sqrt(diff.alphas_cum_prev[i]) * diff.betas[i]
                   ) / (1 - diff.alphas_cum[i])
        coef_x  = (torch.sqrt(diff.alphas[i]) * (1 - diff.alphas_cum_prev[i])
                   ) / (1 - diff.alphas_cum[i])
        mean = coef_x0 * x0 + coef_x * x
        if i > 0:
            x = mean + torch.sqrt(diff.posterior_var[i]) * torch.randn_like(x)
        else:
            x = mean
        if return_trajectory and (i % snap_every == 0 or i == 0):
            traj.append(x.detach().cpu())
    return (x, traj) if return_trajectory else x


@torch.no_grad()
def ddim_sample(net, n, steps=50, eta=0.0):
    net.eval()
    step_seq = torch.linspace(cfg.timesteps - 1, 0, steps, dtype=torch.long).tolist()
    x = torch.randn(n, cfg.in_channels, cfg.image_size, cfg.image_size,
                    device=DEVICE)
    for i, t_cur in enumerate(step_seq):
        t = torch.full((n,), t_cur, device=DEVICE, dtype=torch.long)
        eps = net(x, t).float()
        a_t = diff.alphas_cum[t_cur]
        a_prev = (diff.alphas_cum[step_seq[i + 1]]
                  if i + 1 < len(step_seq) else torch.tensor(1.0, device=DEVICE))
        x0 = ((x - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)).clamp(-1, 1)
        # Re-derive eps from clamped x0 to keep update self-consistent
        eps = (x - torch.sqrt(a_t) * x0) / torch.sqrt(1 - a_t)
        sigma = eta * torch.sqrt((1 - a_prev) / (1 - a_t) * (1 - a_t / a_prev))
        dir_xt = torch.sqrt((1 - a_prev - sigma ** 2).clamp(min=0)) * eps
        x = torch.sqrt(a_prev) * x0 + dir_xt + sigma * torch.randn_like(x)
    return x


## Training

In [ ]:
history = {"step_loss": [], "epoch_loss": [], "lr": [], "gpu_mem": [], "epoch_time": []}
best_loss = float("inf")
t_start = time.time()

for epoch in range(start_epoch + 1, cfg.epochs + 1):
    model.train()
    ep_losses = []
    ep_t0 = time.time()
    pbar = tqdm(loader, desc=f"Epoch {epoch:03d}/{cfg.epochs}", leave=False,
                dynamic_ncols=True, mininterval=0.5)
    for x0 in pbar:
        x0 = x0.to(DEVICE, non_blocking=True)
        t = torch.randint(0, cfg.timesteps, (x0.size(0),), device=DEVICE, dtype=torch.long)
        with autocast(enabled=cfg.use_amp, dtype=AMP_DTYPE):
            xt, noise = diff.q_sample(x0, t)
            pred = model(xt, t)
            loss = F.mse_loss(pred, noise)
        optimizer.zero_grad(set_to_none=True)
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
        scheduler.step()
        ema.update(model)
        v = loss.item()
        history["step_loss"].append(v); history["lr"].append(scheduler.get_last_lr()[0])
        ep_losses.append(v)
        pbar.set_postfix(loss=f"{v:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}",
                         mem=f"{gpu_mem_mb():.0f}MB")

    ep_loss = float(np.mean(ep_losses))
    ep_time = time.time() - ep_t0
    history["epoch_loss"].append(ep_loss); history["gpu_mem"].append(gpu_mem_mb())
    history["epoch_time"].append(ep_time)
    print(f"[{epoch:03d}/{cfg.epochs}] loss={ep_loss:.4f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}  mem={gpu_mem_mb():.0f}MB  time={ep_time:.1f}s")

    if epoch % cfg.sample_every == 0 or epoch == cfg.epochs:
        imgs = ddim_sample(ema.shadow, cfg.n_samples, steps=cfg.ddim_steps)
        show_grid(imgs, nrow=4, title=f"EMA samples - epoch {epoch}",
                  save=str(EPOCH_DIR / f"epoch_{epoch:03d}.png"), figsize=(8, 8))

    save_checkpoint(CKPT_DIR / "last.pt", epoch, ep_loss)
    if ep_loss < best_loss:
        best_loss = ep_loss
        save_checkpoint(CKPT_DIR / "best.pt", epoch, ep_loss)

save_checkpoint(CKPT_DIR / "final.pt", cfg.epochs, history["epoch_loss"][-1])
total_t = time.time() - t_start
print(f"\nTraining complete in {total_t/60:.2f} min")


## Loss Curve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
sl = np.array(history["step_loss"])
ax[0].plot(sl, color="#ff6ec7", alpha=0.3, label="step")
if len(sl) > 50:
    k = max(50, len(sl) // 200)
    smooth = np.convolve(sl, np.ones(k) / k, mode="valid")
    ax[0].plot(np.arange(len(smooth)) + k - 1, smooth, color="#ffe066", linewidth=2,
               label=f"EMA({k})")
ax[0].set_title("Training Loss"); ax[0].set_xlabel("step"); ax[0].legend()
ax[1].plot(history["lr"], color="#7df9ff", linewidth=2); ax[1].set_title("Learning Rate")
ax[1].set_xlabel("step")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "loss_curve.png", dpi=150, bbox_inches="tight"); plt.show()


## DDPM Sampling

In [ ]:
samples_ddpm, traj = ddpm_sample(ema.shadow, 8, return_trajectory=True)
show_grid(samples_ddpm, nrow=8, title="DDPM (1000 steps) - EMA",
          save=str(ASSETS_DIR / "ddpm_samples.png"), figsize=(16, 3))


## DDIM Sampling

In [ ]:
samples_ddim = ddim_sample(ema.shadow, 64, steps=cfg.ddim_steps)
show_grid(samples_ddim, nrow=8, title=f"DDIM ({cfg.ddim_steps} steps) - EMA",
          save=str(ASSETS_DIR / "ddim_samples.png"), figsize=(12, 12))


## Reverse Trajectory

In [ ]:
strip = torch.cat([t[:6] for t in traj], dim=0)
show_grid(strip, nrow=6, title="Reverse Diffusion Trajectory  (x_T -> x_0)",
          save=str(TRAJ_DIR / "trajectory_grid.png"),
          figsize=(12, 2 * len(traj)))


## Sample Grid + Individual Saves

In [ ]:
big = ddim_sample(ema.shadow, 64, steps=cfg.ddim_steps)
show_grid(big, nrow=8, title="Final Sample Grid (8x8) - EMA + DDIM",
          save=str(ASSETS_DIR / "samples_grid.png"), figsize=(14, 14))

# Save individual generated images for downstream use
for i in range(big.size(0)):
    img = (denorm(big[i].float().cpu()).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    imageio.imwrite(SAMPLES_DIR / f"butterfly_{i:03d}.png", img)
print(f"{big.size(0)} individual samples saved -> {SAMPLES_DIR}")


## Raw Model Sample Grid

In [ ]:
# Backup grid sampled from the LIVE model (no EMA), useful when EMA decay
# was too high relative to training length and the shadow under-tracked.
raw_big = ddim_sample(model, 64, steps=cfg.ddim_steps)
show_grid(raw_big, nrow=8, title="Final Sample Grid (8x8) - RAW model + DDIM",
          save=str(ASSETS_DIR / "samples_grid_raw.png"), figsize=(14, 14))


## Latent Interpolation

In [ ]:
def slerp(z1, z2, t):
    z1n = z1 / z1.norm(dim=1, keepdim=True)
    z2n = z2 / z2.norm(dim=1, keepdim=True)
    omega = torch.acos((z1n * z2n).sum(dim=1, keepdim=True).clamp(-1, 1))
    so = torch.sin(omega)
    return (torch.sin((1 - t) * omega) / so) * z1 + (torch.sin(t * omega) / so) * z2


@torch.no_grad()
def interpolate(n_pairs=4, steps=8):
    rows = []
    step_seq = torch.linspace(cfg.timesteps - 1, 0, cfg.ddim_steps, dtype=torch.long).tolist()
    for _ in range(n_pairs):
        z1 = torch.randn(1, cfg.in_channels, cfg.image_size, cfg.image_size, device=DEVICE)
        z2 = torch.randn(1, cfg.in_channels, cfg.image_size, cfg.image_size, device=DEVICE)
        zs = torch.cat([
            slerp(z1.flatten(1), z2.flatten(1),
                  torch.tensor([[float(a)]], dtype=torch.float32, device=DEVICE)).view_as(z1)
            for a in np.linspace(0, 1, steps)
        ], dim=0)
        x = zs
        for i, t_cur in enumerate(step_seq):
            t = torch.full((x.size(0),), t_cur, device=DEVICE, dtype=torch.long)
            eps = ema.shadow(x, t).float()
            a_t = diff.alphas_cum[t_cur]
            a_prev = (diff.alphas_cum[step_seq[i + 1]]
                      if i + 1 < len(step_seq) else torch.tensor(1.0, device=DEVICE))
            x0 = ((x - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)).clamp(-1, 1)
            eps = (x - torch.sqrt(a_t) * x0) / torch.sqrt(1 - a_t)
            x = torch.sqrt(a_prev) * x0 + torch.sqrt((1 - a_prev).clamp(min=0)) * eps
        rows.append(x.cpu())
    return torch.cat(rows, dim=0)


inter = interpolate(4, 8)
show_grid(inter, nrow=8, title="Latent Spherical Interpolation",
          save=str(ASSETS_DIR / "interpolation.png"), figsize=(14, 7))


## Forward vs Reconstructed

In [ ]:
@torch.no_grad()
def forward_vs_reconstructed(real, t_val=400):
    real = real.to(DEVICE)
    t = torch.full((real.size(0),), t_val, device=DEVICE, dtype=torch.long)
    noisy, _ = diff.q_sample(real, t)
    x = noisy
    for i in range(t_val, -1, -1):
        ti = torch.full((x.size(0),), i, device=DEVICE, dtype=torch.long)
        eps = ema.shadow(x, ti).float()
        sac  = diff.sqrt_alphas_cum[i]
        smac = diff.sqrt_one_minus_alphas_cum[i]
        x0 = ((x - smac * eps) / sac).clamp(-1, 1)
        coef_x0 = (torch.sqrt(diff.alphas_cum_prev[i]) * diff.betas[i]
                   ) / (1 - diff.alphas_cum[i])
        coef_x  = (torch.sqrt(diff.alphas[i]) * (1 - diff.alphas_cum_prev[i])
                   ) / (1 - diff.alphas_cum[i])
        mean = coef_x0 * x0 + coef_x * x
        if i > 0:
            x = mean + torch.sqrt(diff.posterior_var[i]) * torch.randn_like(x)
        else:
            x = mean
    panel = torch.cat([real[:8].cpu(), noisy[:8].cpu(), x[:8].cpu()], dim=0)
    show_grid(panel, nrow=8, title=f"Real | Noisy(t={t_val}) | Reconstructed",
              save=str(ASSETS_DIR / "forward_vs_recon.png"), figsize=(16, 6))


forward_vs_reconstructed(next(iter(loader))[:8])


## Feature Maps

In [ ]:
@torch.no_grad()
def feature_maps(t_val=500, n=8):
    x = torch.randn(1, cfg.in_channels, cfg.image_size, cfg.image_size,
                    device=DEVICE).to(memory_format=torch.channels_last)
    t = torch.tensor([t_val], device=DEVICE, dtype=torch.long)
    feats = {}

    def hook(_m, _i, o): feats["x"] = o.detach().float().cpu()

    h = ema.shadow.mid[0].register_forward_hook(hook)
    _ = ema.shadow(x, t); h.remove()

    fmap = feats["x"][0, :n]
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.4))
    for i, ax in enumerate(axes):
        ax.imshow(fmap[i], cmap="magma"); ax.axis("off")
        ax.set_title(f"ch{i}", fontsize=10, color="white")
    fig.suptitle(f"Mid-block feature maps  (t={t_val})", color="white", fontsize=14)
    plt.tight_layout()
    plt.savefig(ASSETS_DIR / "feature_maps.png", dpi=150, bbox_inches="tight"); plt.show()


feature_maps()


## Denoising GIF

In [ ]:
def make_gif():
    _, traj_full = ddpm_sample(ema.shadow, 4, return_trajectory=True)
    frames = []
    for snap in traj_full:
        grid = make_grid(denorm(snap.float()), nrow=4, padding=2)
        arr = (grid.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
        frames.append(arr)
    out = GIFS_DIR / "denoising.gif"
    imageio.mimsave(out, frames, duration=0.15, loop=0)
    # Also expose at top level of /kaggle/working/ for quick download
    shutil.copy2(out, KAGGLE_ROOT / "denoising.gif")
    print(f"GIF saved -> {out}")
    return out


make_gif()


## Training Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.25)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history["epoch_loss"], color="#ff6ec7", linewidth=2.5)
ax1.set_title("Epoch Loss"); ax1.set_xlabel("epoch")

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history["lr"], color="#7df9ff", linewidth=2)
ax2.set_title("Learning Rate"); ax2.set_xlabel("step")

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(history["gpu_mem"], color="#ffe066", marker="o", linewidth=2)
ax3.set_title("GPU Memory (MB)"); ax3.set_xlabel("epoch")

ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(history["epoch_time"], color="#9ef0a0", marker="s", linewidth=2)
ax4.set_title("Epoch Time (s)"); ax4.set_xlabel("epoch")

ax5 = fig.add_subplot(gs[1, 1:])
show = ddim_sample(ema.shadow, 16, steps=cfg.ddim_steps)
g = make_grid(denorm(show.float().cpu()), nrow=8, padding=2)
ax5.imshow(g.permute(1, 2, 0).numpy()); ax5.axis("off"); ax5.set_title("EMA Samples")

fig.suptitle("Butterfly Diffusion - Training Dashboard", fontsize=20, color="white", y=0.98)
plt.savefig(ASSETS_DIR / "dashboard.png", dpi=150, bbox_inches="tight")
plt.savefig(ASSETS_DIR / "banner.png",   dpi=150, bbox_inches="tight")
plt.show()


## Manifest

In [ ]:
manifest = {
    "mode": MODE,
    "config": asdict(cfg),
    "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "final_loss": history["epoch_loss"][-1],
    "best_loss": best_loss,
    "epochs_run": cfg.epochs,
    "total_minutes": round(sum(history["epoch_time"]) / 60, 2),
    "gpu": torch.cuda.get_device_name(0),
    "amp_dtype": str(AMP_DTYPE),
    "torch_version": torch.__version__,
}
with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))


## ZIP Export

## Slim Inference Checkpoint

In [ ]:
# Re-save EMA-only weights for lightweight inference / Streamlit deployment.
# Drops optimizer + scheduler + scaler state, keeps EMA + config.
ema_only_path = CKPT_DIR / "ema_only.pt"
torch.save({
    "ema": ema.state_dict(),
    "config": asdict(cfg),
    "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "final_loss": history["epoch_loss"][-1],
}, ema_only_path)
sz_mb = ema_only_path.stat().st_size / 1e6
print(f"Slim inference checkpoint: {ema_only_path}  ({sz_mb:.1f} MB)")


In [ ]:
# Single-archive download bundle for the entire run.
ZIP_PATH = KAGGLE_ROOT / "butterfly_diffusion_deliverables.zip"

def add_tree(zf, base: Path, arcroot: str):
    if not base.exists(): return 0
    n = 0
    for f in base.rglob("*"):
        if f.is_file():
            zf.write(f, arcname=f"{arcroot}/{f.relative_to(base)}")
            n += 1
    return n

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    totals = {
        "checkpoints":       add_tree(zf, CKPT_DIR,    "checkpoints"),
        "outputs":           add_tree(zf, OUT_DIR,     "outputs"),
        "assets":            add_tree(zf, ASSETS_DIR,  "assets"),
        "generated_samples": add_tree(zf, SAMPLES_DIR, "generated_samples"),
    }
    zf.writestr("manifest.json", json.dumps(manifest, indent=2))

print("Archive contents:")
for k, v in totals.items(): print(f"  {k:<20} {v} files")
print(f"\nArchive size : {ZIP_PATH.stat().st_size/1e6:.2f} MB")
print(f"Archive path : {ZIP_PATH}")
print("\nDownload it from the Kaggle Notebook's Output tab.")


## Deliverables

In [ ]:
# Final inventory of everything ready to download from /kaggle/working/.
print("=" * 70)
print(" KAGGLE DELIVERABLES")
print("=" * 70)

def size_mb(p: Path):
    if p.is_file(): return p.stat().st_size / 1e6
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6

for name, p in [
    ("Full archive (ZIP)",   ZIP_PATH),
    ("Checkpoints",          CKPT_DIR),
    ("Assets (README imgs)", ASSETS_DIR),
    ("Outputs (epoch grids)", OUT_DIR),
    ("Generated samples",    SAMPLES_DIR),
    ("Denoising GIF",        KAGGLE_ROOT / "denoising.gif"),
    ("Slim EMA checkpoint",  CKPT_DIR / "ema_only.pt"),
]:
    if p.exists():
        kind = "file" if p.is_file() else "dir "
        print(f"  [{kind}] {str(p):<50}  {size_mb(p):>8.2f} MB")
print("=" * 70)
print("Tip: in the Kaggle Notebook UI, open the 'Output' panel on the right")
print("     and click 'Download' to grab any artifact, or download the ZIP")
print("     for the full bundle.")
